# **Import Librarys**

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
from datasets import Dataset

# **Tokenizer dan Dataset Format (untuk HuggingFace Trainer)**

In [5]:
# Load tokenizer
model_name = "indobenchmark/indobert-base-p2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load data yang telah diproses
train_df = pd.read_csv(r'C:\Users\user\Documents\dataset\split_data_half\train_dataset.csv')
val_df = pd.read_csv(r'C:\Users\user\Documents\dataset\split_data_half\val_dataset.csv')
test_df = pd.read_csv(r'C:\Users\user\Documents\dataset\split_data_half\test_dataset.csv')

# Konversi ke Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Fungsi tokenisasi
def tokenize_function(examples):
    return tokenizer(examples['normalized'], truncation=True,max_length=256)

# Tokenisasi dataset
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/12364 [00:00<?, ? examples/s]

Map:   0%|          | 0/1766 [00:00<?, ? examples/s]

Map:   0%|          | 0/3534 [00:00<?, ? examples/s]

# **Set format dataset ke format PyTorch**

In [6]:
train_dataset = train_dataset.rename_column("label_num", "labels")
val_dataset = val_dataset.rename_column("label_num", "labels")
test_dataset = test_dataset.rename_column("label_num", "labels")

# set format dataset untuk PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# **Model Setup**

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

training_args = TrainingArguments(
    output_dir="../training_args",
    eval_strategy="epoch",  # Evaluasi dilakukan di akhir setiap epoch
    save_strategy="epoch",  # Simpan model di akhir setiap epoch
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy'
)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# **Define metric evaluasi**

In [8]:
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)['accuracy'],
        "f1": f1.compute(predictions=predictions, references=labels, average='macro')['f1']
    }


# **Trainer Setup**

In [9]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

# **Fine-tuning IndoBERT**

In [10]:
# untuk mempercepat training
%pip install accelerate -U

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Mulai mencatat waktu
start_time = time.time()

# Melatih model
trainer.train()

# Selesai mencatat waktu
end_time = time.time()

# Menghitung durasi
training_duration = end_time - start_time

# Tampilkan durasi dalam format jam:menit:detik
import datetime
formatted_duration = str(datetime.timedelta(seconds=int(training_duration)))
print(f"Durasi pelatihan: {formatted_duration}")

C:\Users\user\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


# **Evaluasi Model**

In [ ]:
results = trainer.evaluate(test_dataset)
print(results)

# **save model**

In [ ]:
# Simpan model
trainer.save_model(r"C:\Users\user\Documents\NLP\models\indoBERT_finetuning_half")

# **test model**

In [ ]:
preds = trainer.predict(test_dataset)
pred_labels = preds.predictions.argmax(-1)

In [ ]:
from sklearn.metrics import classification_report

target_names = ['positif', 'netral', 'negatif']
print(classification_report(test_dataset["labels"], pred_labels, target_names=target_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_dataset['labels'], pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['positive', 'neutral', 'negative'], yticklabels=['positive', 'neutral', 'negative'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
def predict_sentiment_debug(text):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Check if CUDA is available
    model.to(device)  # Move model to the appropriate device
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)  # Move inputs to the same device
    outputs = model(**inputs)
    probs = softmax(outputs.logits, dim=-1)  # Hitung probabilitas
    label_map = {0: "positive", 1: "neutral", 2: "negative"}  # Mapping label
    for i, prob in enumerate(probs[0]):
        print(f"{label_map[i]}: {prob.item() * 100:.3f}%")
    predicted_class = torch.argmax(probs, dim=-1).item()  # Ambil label prediksi
    confidence = probs[0][predicted_class].item() * 100  # Ambil probabilitas prediksi
    return label_map[predicted_class], confidence

# Contoh teks untuk prediksi
text = "cantik"
label, confidence = predict_sentiment_debug(text)

# Output hasil prediksi
print(f"Text: {text} | Label: {label} ({confidence:.3f}%)")